# 04 Subhalo smoke test

A few minutes on tiny models to check the new training code before the real subhalo runs:

1. A model conditioned on tracer spacing (FiLM) starts out giving exactly the same output as the plain model it was copied from.
2. A tiny run with two subhalo spacings, FiLM, a curriculum and `init_from` trains, resumes and scores.
3. The spacing sweep runs.

Scores will be poor; that is expected.

Runtime: **L4 GPU** with **High-RAM**. Needs the `GITHUB_TOKEN` secret.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
del token
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. FiLM starts as the identity

In [ ]:
import numpy as np
from deepvoid.models import copy_weights, unet3d

plain = unet3d(depth=3, base_filters=8)
film = unet3d(depth=3, base_filters=8, conditioning={'units': 16, 'noise': 0.1})
copied, skipped = copy_weights(plain, film)
print(len(copied), 'layers copied;', len(skipped), 'left as built')

x = np.random.default_rng(0).normal(size=(2, 32, 32, 32, 1)).astype('float32')
s = np.full((2, 1, 1, 1, 1), 7.0, 'float32')
diff = np.abs(plain.predict_on_batch(x) - film.predict_on_batch({'density': x, 'spacing': s})).max()
print('max difference:', diff)
assert diff < 1e-5

## 2. Tiny run: two spacings, FiLM, curriculum, started from a saved plain model

In [ ]:
import shutil
from pathlib import Path
from deepvoid.train import run

shutil.rmtree('/content/runs', ignore_errors=True)
Path('/content/runs').mkdir()
plain.save('/content/runs/plain.keras')

config = {
    'name': 'smoke_film',
    'output': '/content/runs',
    'data': {'inputs': ['tng300-3/subhalos/L3', 'tng300-3/subhalos/L10'],
             'labels': 'tng300-3/mask'},
    'model': {'depth': 3, 'base_filters': 8, 'conditioning': {'units': 16, 'noise': 0.1}},
    'training': {'crop': 64, 'batch_size': 4, 'steps_per_epoch': 20, 'val_batches': 2,
                 'epochs': 2, 'init_from': 'plain.keras',
                 'curriculum': {'start': [3, 1], 'end': [1, 1], 'epochs': 2},
                 'loss': {'name': 'scce', 'class_weights': 'auto'}},
    'evaluate': {'inputs': ['tng300-3/subhalos/L3', 'tng300-3/subhalos/L5',
                            'tng300-3/subhalos/L10']},
}
out = run(config)

The log above should show `initialized ... layers`, and `test_scores.json` should have three entries, including L5, which the model never trained on.

## 3. Resume for one more epoch

This should print `resuming` and start at epoch 2. The log should have `val_mcc_L3` and `val_mcc_L10` columns.

In [ ]:
config['training']['epochs'] = 3
out = run(config)
print(open(out / 'log.csv').read())

## 4. Spacing sweep

In [ ]:
from deepvoid.train import spacing_sweep
sweep = spacing_sweep(out, 'tng300-3/subhalos/L10', [3, 10])

Paste the output of the cells (or any error) back to Claude.